# Set up

In [14]:
import os
import numpy as np
import mne

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks, constraints
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Reshape, Conv2D, BatchNormalization, Activation, AveragePooling2D, Flatten, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import max_norm

import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import time

In [23]:
class Config:
    BASE_PATH = 'C:/ESC5Work/ProjectM5/files'
    SAMPLING_RATE = 160 # Hz
    # RESAMPLE = 128
    N_CHANNELS = 14 # Number of EEG channels
    N_CLASSES = 2 # Left vs Right hand imagery

    ALL_SUBJECTS = list(range(1, 110))
    # # EEGNet
    TRAIN_SUBJECTS = list(range(1, 78)) # Subjects 1-77 for training (~70%)
    VALID_SUBJECTS = list(range(78, 94)) # Subjects 78-93 for validation (~15%)
    TEST_SUBJECTS = list(range(94, 110)) # Subjects 94-109 for final testing (~15%)
    
    # TRAIN_SUBJECTS = list(range(1, 88)) # Subjects 1-87 for training (~80%)
    # VALID_SUBJECTS = list(range(88, 99)) # Subjects 88-98 for validation (~15%)
    # TEST_SUBJECTS = list(range(99, 110)) # Subjects 99-109 for final testing (~15%)
    
    # SVM
    TRAIN_SVM_SUBJECTS = list(range(1, 88)) # Subjects 1-87 for training (approx. 80%)
    TEST_SVM_SUBJECTS = list(range(88, 110)) # Subjects 88-109 for final testing (approx. 20%)

    # Signal Processing Parameters
    LOWCUT = 8.0 # Lower cutoff frequency (Hz) for Mu 8-12 rhythm
    HIGHCUT = 30.0 # Upper cutoff frequency (Hz) for Beta 13-30 rhythm
    WINDOW_SIZE = 4.0 # Time window in seconds
    T_MIN = 0.0 # Start time of the epoch relative to the event
    T_MAX = 4.0 # End time of the epoch

    # EEGNet Architecture Parameters
    F1 = 8 # Number of temporal filters
    D = 2 # Number of spatial filters
    F2 = 16 # Number of pointwise filters (D*F1)
    KERNEL_LENGTH = 64 # Length of the temporal convolution kernel
    DROPOUT_RATE = 0.5 # Dropout rate for regularization

    # Training Parameters
    BATCH_SIZE = 16 # Smaller batch size for better generalization on EEG data
    EPOCHS = 200 # Max epochs, with early stopping
    LEARNING_RATE = 0.0005 #0.0005
    EARLY_STOPPING_PATIENCE = 30 # Increased patience for EEGNet

    PLOTLY_TEMPLATE = 'plotly_dark'
    
    SELECTED_CHANNELS = ['Fc5.', 'Fc6.', 'Af3.', 'Af4.', 'F7..', 'F3..', 'F4..', 'F8..', 'T7..', 'T8..', 'P7..', 'P8..', 'O1..', 'O2..']

config = Config()

In [16]:
import warnings
warnings.filterwarnings(
    "ignore", 
    message=".*All epochs were dropped.*", 
    category=RuntimeWarning
)

In [17]:
def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    labels = ['Left Hand', 'Right Hand']
    fig = ff.create_annotated_heatmap(z=cm, x=labels, y=labels, colorscale='Greens')
    fig.update_layout(title_text="Confusion Matrix (Final Test Set)",
                    xaxis_title="Predicted Label", yaxis_title="True Label",
                    template=config.PLOTLY_TEMPLATE)
    fig.show()

In [19]:
def pred_one(pred, real_num):
    labels = ['Left Hand Signal', 'Right Hand Signal']
    
    print(pred)
    print('\n-----------------------------------\n')
    
    pred_num = 1
    if pred[0][0]>pred[0][1]:
        pred_num = 0
    print('Predicted Class : ', labels[pred_num])
    print(f'Accuracy : {pred[0][pred_num]*100:.2f}%')
    
    print('Real Class : ', labels[real_num])

# EEGNet

## Preprocessing

In [21]:
def load_and_preprocess_subject_data(subject_id, base_path):
    mne.set_log_level("ERROR")
    mi_runs = [4, 8, 12]

    raw_files = []
    for run_id in mi_runs:
        subject_str = f"S{subject_id:03d}"
        run_str = f"R{run_id:02d}"
        file_path = f"{base_path}/{subject_str}/{subject_str}{run_str}.edf"

        if os.path.exists(file_path):
            try:
                raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
                raw_files.append(raw)
            except Exception as e:
                print(f"-- error to load {file_path}: {e}")
        else:
            pass

    if not raw_files:
        return None, None

    raw_concat = mne.concatenate_raws(raw_files, verbose=False)

    raw_concat.filter(config.LOWCUT, config.HIGHCUT, fir_design='firwin', skip_by_annotation='edge', verbose=False)
    raw_concat.notch_filter(freqs=50, verbose=False)
    # raw_concat.resample(config.RESAMPLE, npad="auto")
    raw_concat.set_eeg_reference('average', projection=False)
    raw_concat.apply_proj(verbose=False)

    events, event_id = mne.events_from_annotations(raw_concat, verbose=False)
    # print(events)

    # picks = mne.pick_types(raw_concat.info, meg=False, eeg=True, stim=False, eog=False, exclude='bads')
    picks = mne.pick_channels(raw_concat.info["ch_names"], include=config.SELECTED_CHANNELS)
    
    # Epoching with rejection
    reject_criteria = dict(eeg=200e-6)  
    flat_criteria = dict(eeg=1e-6)
    
    try:
        rest_epochs = mne.Epochs(raw_concat, events, dict(rest=1), 
                         tmin=config.T_MIN, tmax=config.T_MAX,
                         picks=picks, baseline=None, preload=True,
                         reject=reject_criteria, flat=flat_criteria,
                         on_missing='warn')
        epochs = mne.Epochs(raw_concat, events, dict(left_hand=2, right_hand=3),
                            tmin=config.T_MIN, tmax=config.T_MAX, proj=True,
                            picks=picks, baseline=None, preload=True, verbose=False, reject=reject_criteria, flat=flat_criteria,
                            on_missing='warn')
    except ValueError:
        return None, None
    
    if len(epochs)<=0:
        return None, None
    
    rest_data = rest_epochs.get_data()
    rest_mean = rest_data.mean(axis=(0, 2))

    # Normalize data - online vs offline
    X = epochs.get_data()
    X = X - rest_mean[:, np.newaxis] # baseline correction
    X = (X - X.mean(axis=-1, keepdims=True)) / X.std(axis=-1, keepdims=True)

    labels = epochs.events[:, -1] - 2  # Map labels to 0 (left) and 1 (right)

    return X, labels

In [8]:
def load_dataset_for_subjects(subject_list, base_path):
    all_epochs_list = []
    all_labels_list = []
    expected_n_samples = 641

    for subject_id in subject_list:
        print(f"Processing {subject_id:03d}...", end=" -> ")
        epochs, labels = load_and_preprocess_subject_data(subject_id, base_path)

        if epochs is not None and len(epochs) > 0:
            if expected_n_samples==None:
                expected_n_samples = epochs.shape[2]
            if expected_n_samples is None:
                expected_n_samples = epochs.shape[2]
            if epochs.shape[2] != expected_n_samples:
                print(f"-- Wrong length ({epochs.shape[2]}). Skipping subject.")
                continue

            all_epochs_list.append(epochs)
            all_labels_list.append(labels)
            print(f"Finish {len(epochs)} trials.")
        else:
            print("-- No valid trials found. (dropped all)")

    if not all_epochs_list:
        return np.array([]), np.array([])

    return np.concatenate(all_epochs_list, axis=0), np.concatenate(all_labels_list, axis=0)

In [24]:
print("\n--- Loading Training Data ---")
X_train, y_train = load_dataset_for_subjects(config.TRAIN_SUBJECTS, config.BASE_PATH)

print("\n--- Loading Validation Data ---")
X_val, y_val = load_dataset_for_subjects(config.VALID_SUBJECTS, config.BASE_PATH)

if X_train.size > 0 and X_val.size > 0:
    print("\n Data Summary >> ")
    print(f" Training data shape: {X_train.shape}")
    print(f" Training labels shape: {y_train.shape} (Left: {np.sum(y_train == 0)}, Right: {np.sum(y_train == 1)})\")")
    print(f" Validation data shape: {X_val.shape}")
    print(f" Validation labels shape: {y_val.shape} (Left: {np.sum(y_val == 0)}, Right: {np.sum(y_val == 1)})\")")
else:
    print("\n-- Error")


--- Loading Training Data ---
Processing 001... -> [[    0     0     1]
 [  672     0     3]
 [ 1328     0     1]
 [ 2000     0     2]
 [ 2656     0     1]
 [ 3328     0     2]
 [ 3984     0     1]
 [ 4656     0     3]
 [ 5312     0     1]
 [ 5984     0     3]
 [ 6640     0     1]
 [ 7312     0     2]
 [ 7968     0     1]
 [ 8640     0     3]
 [ 9296     0     1]
 [ 9968     0     2]
 [10624     0     1]
 [11296     0     3]
 [11952     0     1]
 [12624     0     2]
 [13280     0     1]
 [13952     0     2]
 [14608     0     1]
 [15280     0     3]
 [15936     0     1]
 [16608     0     2]
 [17264     0     1]
 [17936     0     3]
 [18592     0     1]
 [19264     0     2]
 [20000     0     1]
 [20672     0     2]
 [21328     0     1]
 [22000     0     3]
 [22656     0     1]
 [23328     0     2]
 [23984     0     1]
 [24656     0     3]
 [25312     0     1]
 [25984     0     2]
 [26640     0     1]
 [27312     0     3]
 [27968     0     1]
 [28640     0     2]
 [29296     0     1]
 [2

KeyboardInterrupt: 

## Modeling

In [10]:
def prep_eegnetDATA(X_train, y_train, X_val, y_val, augment = False):
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    
    scaler = StandardScaler()
    X_train_FitFlat = scaler.fit_transform(X_train_flat)
    X_val_FitFlat = scaler.fit_transform(X_val_flat)
    
    X_train_fit = X_train_FitFlat.reshape(X_train.shape)
    X_val_fit = X_val_FitFlat.reshape(X_val.shape)
    
    # source - reason
    if augment:
        noise_factor = 0.1 # 10%
        X_train_aug = X_train_fit + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=X_train_fit.shape)
        # y_train_aug = y_train
        
        X_train_aug = np.concatenate((X_train_fit, X_train_aug), axis=0)
        y_train_aug = np.concatenate((y_train, y_train), axis=0)
    else:
        X_train_aug, y_train_aug = X_train_fit, y_train

    X_train_fin = X_train_aug[:, :, :, np.newaxis]
    X_val_fin = X_val_fit[:, :, :, np.newaxis]
    y_train_fin = to_categorical(y_train_aug, config.N_CLASSES)
    y_val_fin = to_categorical(y_val, config.N_CLASSES)
    
    return X_train_fin, y_train_fin, X_val_fin, y_val_fin, scaler

In [11]:
X_train_fin, y_train_fin, X_val_fin, y_val_fin, scaler = prep_eegnetDATA(X_train, y_train, X_val, y_val, augment=True)

print(f'Train shape (X): {X_train_fin.shape}')
print(f'Validation shape (X): {X_val_fin.shape}')
# trial, channel, sample

Train shape (X): (5514, 14, 641, 1)
Validation shape (X): (530, 14, 641, 1)


## Baseline

In [30]:
# Example parameters
input_shape = (config.N_CHANNELS, X_train_fin.shape[2], 1)
num_classes = config.N_CLASSES

# Model
inputs = layers.Input(shape=input_shape, name='input_layer')
x = layers.Flatten(name='flatten_layer')(inputs)
outputs = layers.Dense(num_classes, activation='sigmoid', name='output_layer')(x)

baseline_model = models.Model(inputs=inputs, outputs=outputs, name='simple_flat_dense_model')

# Compile the model
optimizer = optimizers.Adam(learning_rate=0.1)
baseline_model.compile(optimizer=optimizer,
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Show summary
baseline_model.summary()

baseline_model.save("simple_flat_dense_model.keras")

Model: "simple_flat_dense_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 14, 641, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_layer (Flatten)         │ (None, 8974)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 2)              │        17,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,950 (70.12 KB)

 Trainable params: 17,950 (70.12 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
training_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=config.EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
        mode='max'
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=15, # Increased patience for learning rate reduction
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'simple_flat_dense_model.keras',
        monitor='val_accuracy', 
        save_best_only=True,
        verbose=1,
        mode='max'
    )
]

history = baseline_model.fit( # fit the EEGNet model
    X_train_fin,
    y_train_fin,
    batch_size=config.BATCH_SIZE,
    epochs=config.EPOCHS,
    validation_data=(X_val_fin, y_val_fin),
    callbacks=training_callbacks,
    verbose=1,
    shuffle=True
)

NameError: name 'baseline_model' is not defined

In [39]:
# --- Plot Training History ---
def plot_training_history(history):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Model Accuracy", "Model Loss"))

    # Accuracy Plot
    fig.add_trace(go.Scatter(y=history.history['accuracy'], name='Train Accuracy', mode='lines+markers'), row=1, col=1)
    fig.add_trace(go.Scatter(y=history.history['val_accuracy'], name='Validation Accuracy', mode='lines+markers'), row=1, col=1)

    # Loss Plot
    fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss', mode='lines+markers'), row=1, col=2)
    fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Validation Loss', mode='lines+markers'), row=1, col=2)

    fig.update_layout(title_text="Model Training History", template=config.PLOTLY_TEMPLATE, height=400)
    fig.update_xaxes(title_text="Epoch")
    fig.update_yaxes(title_text="Metric Value")
    fig.show()

print("\n--- Training History ---")
plot_training_history(history)

# --- Validation Set Evaluation ---
print("\n--- Evaluating Performance on Val Set ---")
best_model = models.load_model('simple_flat_dense_model.keras')
y_pred_val = best_model.predict(X_val_fin, batch_size=config.BATCH_SIZE)
y_pred_classes = np.argmax(y_pred_val, axis=1)
y_true_classes = np.argmax(y_val_fin, axis=1)

# Classification Report
print("\nClassification Report (Val Set):")
class_report = classification_report(y_true_classes, y_pred_classes, target_names=['Left Hand', 'Right Hand'])
print(class_report)

print("\n--- Confusion Matrix ---")
plot_confusion_matrix(y_true_classes, y_pred_classes)


--- Training History ---



--- Evaluating Performance on Val Set ---
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

Classification Report (Val Set):
              precision    recall  f1-score   support

   Left Hand       0.50      0.50      0.50       266
  Right Hand       0.49      0.49      0.49       264

    accuracy                           0.50       530
   macro avg       0.50      0.50      0.50       530
weighted avg       0.50      0.50      0.50       530


--- Confusion Matrix ---


In [40]:
print("\nLoading test data -> ")
X_test, y_test = load_dataset_for_subjects(config.TEST_SUBJECTS, config.BASE_PATH)

print(f"Success. Found {X_test.shape[0]} trials.")
print(f"\nTest data shape: {X_test.shape}")
print(f", Test labels shape: {y_test.shape} (Left: {np.sum(y_test == 0)}, Right: {np.sum(y_test == 1)})")

print("\nPreparing test data ->")
X_test_flat = X_test.reshape(X_test.shape[0], -1)
X_test_scaled_flat = scaler.transform(X_test_flat)  # IMPORTANT: Only transform, do not fit again
X_test_scaled = X_test_scaled_flat.reshape(X_test.shape)
X_test_final = X_test_scaled[:, :, :, np.newaxis]
y_test_final = to_categorical(y_test, config.N_CLASSES)
print(f" Final Test Shape: {X_test_final.shape}")

print("\nEvaluating model on the test set :")
test_loss, test_accuracy = best_model.evaluate(X_test_final, y_test_final, verbose=0, batch_size=config.BATCH_SIZE)
print(f"\nFinal Test Accuracy: {test_accuracy*100:.2f}%")
print(f"   Final Test Loss: {test_loss:.4f}")

y_pred_test = best_model.predict(X_test_final, batch_size=config.BATCH_SIZE)
y_pred_test_classes = np.argmax(y_pred_test, axis=1)
y_true_test_classes = y_test

print("\nClassification Report (Test Set):")
print(classification_report(y_true_test_classes, y_pred_test_classes, target_names=['Left Hand', 'Right Hand']))
plot_confusion_matrix(y_true_test_classes, y_pred_test_classes)


Loading test data -> 
Processing 094... -> Finish 26 trials.
Processing 095... -> Finish 7 trials.
Processing 096... -> Finish 18 trials.
Processing 097... -> Finish 45 trials.
Processing 098... -> Finish 45 trials.
Processing 099... -> Finish 3 trials.
Processing 100... -> 

C:\Users\scpwk\AppData\Local\Temp\ipykernel_22532\4062185886.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\scpwk\AppData\Local\Temp\ipykernel_22532\4062185886.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\scpwk\AppData\Local\Temp\ipykernel_22532\4062185886.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.



-- Wrong length (513). Skipping subject.
Processing 101... -> Finish 45 trials.
Processing 102... -> Finish 42 trials.
Processing 103... -> Finish 14 trials.
Processing 104... -> Finish 42 trials.
Processing 105... -> Finish 45 trials.
Processing 106... -> Finish 37 trials.
Processing 107... -> Finish 37 trials.
Processing 108... -> Finish 7 trials.
Processing 109... -> Finish 44 trials.
Success. Found 457 trials.

Test data shape: (457, 14, 641)
, Test labels shape: (457,) (Left: 229, Right: 228)

Preparing test data ->
 Final Test Shape: (457, 14, 641, 1)

Evaluating model on the test set :

Final Test Accuracy: 47.05%
   Final Test Loss: 210.5803
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

Classification Report (Test Set):
              precision    recall  f1-score   support

   Left Hand       0.47      0.43      0.45       229
  Right Hand       0.47      0.51      0.49       228

    accuracy                           0.47       457
   macro avg       0.47      0.47      0.47      

## Train Model

In [12]:
def EEGNet_model(nb_classes, Chans=config.N_CHANNELS, Samples=X_train_fin.shape[2], dropoutRate=0.5,
           kernLength=config.KERNEL_LENGTH, F1=config.F1, D=config.D, F2=config.F2, norm_rate=0.25):
    input_main = layers.Input(shape=(Chans, Samples, 1), name='input_layer')

    # --- Block 1 ---
    # Temporal Convolution
    block1 = layers.Conv2D(F1, (1, kernLength), padding='same', use_bias=False)(input_main)
    block1 = layers.BatchNormalization()(block1)

    # Depthwise Spatial Convolution
    block1 = layers.DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D,
                                     depthwise_constraint=max_norm(1.))(block1)
    block1 = layers.BatchNormalization()(block1)
    block1 = layers.Activation('elu')(block1)
    block1 = layers.AveragePooling2D((1, D))(block1)
    block1 = layers.Dropout(dropoutRate)(block1)

    # --- Block 2 ---
    # Separable Convolution
    block2 = layers.SeparableConv2D(F2, (1, F2), use_bias=False, padding='same')(block1)
    block2 = layers.BatchNormalization()(block2)
    block2 = layers.Activation('elu')(block2)
    block2 = layers.AveragePooling2D((1, F1))(block2)
    block2 = layers.Dropout(dropoutRate)(block2)

    # --- Classification Block ---
    flatten = layers.Flatten(name='flatten')(block2)
    dense = layers.Dense(nb_classes, name='dense', kernel_constraint=max_norm(norm_rate))(flatten)
    sigmoid = layers.Activation('sigmoid', name='sigmoid')(dense)

    return models.Model(inputs=input_main, outputs=sigmoid, name="EEGNet")

# --- Instantiate and Compile the Model ---
input_shape = X_train_fin.shape[1:]

model = EEGNet_model(
    nb_classes=config.N_CLASSES,
    Chans=input_shape[0],
    Samples=input_shape[1],
    dropoutRate=config.DROPOUT_RATE,
    kernLength=config.KERNEL_LENGTH,
    F1=config.F1, D=config.D, F2=config.F2
)

# Compile the model with Adam optimizer
optimizer = optimizers.Adam(learning_rate=config.LEARNING_RATE)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Print the model summary
print("EEGNet Model Summary:")
model.summary()

EEGNet Model Summary:


Model: "EEGNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 14, 641, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 14, 641, 8)     │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 14, 641, 8)     │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 1, 641, 16)     │           224 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 641, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1, 641, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 1, 320, 16)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 320, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 1, 320, 16)     │           512 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 320, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 1, 320, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 1, 40, 16)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 40, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 640)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         1,282 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sigmoid (Activation)            │ (None, 2)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,690 (10.51 KB)

 Trainable params: 2,610 (10.20 KB)

 Non-trainable params: 80 (320.00 B)

In [13]:
training_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=config.EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
        mode='max'
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=config.EARLY_STOPPING_PATIENCE,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_eegnet_model.keras',
        monitor='val_accuracy', 
        save_best_only=True,
        verbose=1,
        mode='max'
    )
]

history = model.fit( # fit the EEGNet model
    X_train_fin,
    y_train_fin,
    batch_size=config.BATCH_SIZE,
    epochs=config.EPOCHS,
    validation_data=(X_val_fin, y_val_fin),
    callbacks=training_callbacks,
    verbose=1,
    shuffle=True
)

Epoch 1/200
345/345 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.5088 - loss: 0.6946
Epoch 1: val_accuracy improved from -inf to 0.48868, saving model to best_eegnet_model.keras
345/345 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - accuracy: 0.5088 - loss: 0.6946 - val_accuracy: 0.4887 - val_loss: 0.6926 - learning_rate: 5.0000e-04
Epoch 2/200
344/345 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.5234 - loss: 0.6911
Epoch 2: val_accuracy improved from 0.48868 to 0.51509, saving model to best_eegnet_model.keras
345/345 ━━━━━━━━━━━━━━━━━━━━ 15s 44ms/step - accuracy: 0.5235 - loss: 0.6911 - val_accuracy: 0.5151 - val_loss: 0.6901 - learning_rate: 5.0000e-04
Epoch 3/200
344/345 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.5556 - loss: 0.6840
Epoch 3: val_accuracy improved from 0.51509 to 0.60566, saving model to best_eegnet_model.keras
345/345 ━━━━━━━━━━━━━━━━━━━━ 16s 45ms/step - accuracy: 0.5557 - loss: 0.6840 - val_accuracy: 0.6057 - val_loss: 0.6758 - learning_rate: 5.0000e-04
Epoch 4/20

KeyboardInterrupt: 

## Evaluation

In [33]:
# --- Plot Training History ---
def plot_training_history(history):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Model Accuracy", "Model Loss"))

    # Accuracy Plot
    fig.add_trace(go.Scatter(y=history.history['accuracy'], name='Train Accuracy', mode='lines+markers'), row=1, col=1)
    fig.add_trace(go.Scatter(y=history.history['val_accuracy'], name='Validation Accuracy', mode='lines+markers'), row=1, col=1)

    # Loss Plot
    fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss', mode='lines+markers'), row=1, col=2)
    fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Validation Loss', mode='lines+markers'), row=1, col=2)

    fig.update_layout(title_text="Model Training History", template=config.PLOTLY_TEMPLATE, height=400)
    fig.update_xaxes(title_text="Epoch")
    fig.update_yaxes(title_text="Metric Value")
    fig.show()

print("\n--- Training History ---")
plot_training_history(history)

# --- Validation Set Evaluation ---
print("\n--- Evaluating Performance on Val Set ---")
best_model = models.load_model('best_eegnet_model.keras')
y_pred_val = best_model.predict(X_val_fin, batch_size=config.BATCH_SIZE)
y_pred_classes = np.argmax(y_pred_val, axis=1)
y_true_classes = np.argmax(y_val_fin, axis=1)

# Classification Report
print("\nClassification Report (Val Set):")
class_report = classification_report(y_true_classes, y_pred_classes, target_names=['Left Hand', 'Right Hand'])
print(class_report)

print("\n--- Confusion Matrix ---")
plot_confusion_matrix(y_true_classes, y_pred_classes)


--- Training History ---



--- Evaluating Performance on Val Set ---
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step

Classification Report (Val Set):
              precision    recall  f1-score   support

   Left Hand       0.75      0.76      0.76       144
  Right Hand       0.77      0.76      0.77       150

    accuracy                           0.76       294
   macro avg       0.76      0.76      0.76       294
weighted avg       0.76      0.76      0.76       294


--- Confusion Matrix ---


In [34]:
print("\nLoading test data : ")
X_test, y_test = load_dataset_for_subjects(config.TEST_SUBJECTS, config.BASE_PATH)

print(f"Success. Found {X_test.shape[0]} trials.")
print(f"\nTest data shape: {X_test.shape}")
print(f", Test labels shape: {y_test.shape} (Left: {np.sum(y_test == 0)}, Right: {np.sum(y_test == 1)})")

print("\nPreparing test data :")
X_test_flat = X_test.reshape(X_test.shape[0], -1)
X_test_scaled_flat = scaler.transform(X_test_flat)  # IMPORTANT: Only transform, do not fit again
X_test_scaled = X_test_scaled_flat.reshape(X_test.shape)
X_test_final = X_test_scaled[:, :, :, np.newaxis]
y_test_final = to_categorical(y_test, config.N_CLASSES)
print(f" Final Test Shape: {X_test_final.shape}")

print("\nEvaluating model on the test set :")
test_loss, test_accuracy = best_model.evaluate(X_test_final, y_test_final, verbose=0, batch_size=config.BATCH_SIZE)
print(f"\nFinal Test Accuracy: {test_accuracy*100:.2f}%")
print(f"   Final Test Loss: {test_loss:.4f}")

y_pred_test = best_model.predict(X_test_final, batch_size=config.BATCH_SIZE)
y_pred_test_classes = np.argmax(y_pred_test, axis=1)
y_true_test_classes = y_test

print("\nClassification Report (Test Set):")
print(classification_report(y_true_test_classes, y_pred_test_classes, target_names=['Left Hand', 'Right Hand']))
plot_confusion_matrix(y_true_test_classes, y_pred_test_classes)


Loading test data : 
Processing 099... -> Finish 3 trials.
Processing 100... -> 

C:\Users\scpwk\AppData\Local\Temp\ipykernel_19300\4062185886.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\scpwk\AppData\Local\Temp\ipykernel_19300\4062185886.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\scpwk\AppData\Local\Temp\ipykernel_19300\4062185886.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.



-- Wrong length (513). Skipping subject.
Processing 101... -> Finish 45 trials.
Processing 102... -> Finish 42 trials.
Processing 103... -> Finish 14 trials.
Processing 104... -> Finish 42 trials.
Processing 105... -> Finish 45 trials.
Processing 106... -> Finish 37 trials.
Processing 107... -> Finish 37 trials.
Processing 108... -> Finish 7 trials.
Processing 109... -> Finish 44 trials.
Success. Found 316 trials.

Test data shape: (316, 14, 641)
, Test labels shape: (316,) (Left: 161, Right: 155)

Preparing test data :
 Final Test Shape: (316, 14, 641, 1)

Evaluating model on the test set :

Final Test Accuracy: 65.51%
   Final Test Loss: 0.5748
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

Classification Report (Test Set):
              precision    recall  f1-score   support

   Left Hand       0.69      0.60      0.64       161
  Right Hand       0.63      0.72      0.67       155

    accuracy                           0.66       316
   macro avg       0.66      0.66      0.65       31

In [123]:
n = int(input('Index number :'))

start_time = time.time()

pred = best_model.predict(X_test_final[n:n+1])

end_time = time.time()

real_num = np.argmax(y_test_final[n])
pred_one(pred, real_num)

fin_time = end_time - start_time
print(f"Time used = {fin_time*1000:.2f} milliseconds")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
[[0.7688869  0.23116505]]

-----------------------------------

Predicted Class :  Left Hand Signal
Accuracy : 76.89%
Real Class :  Right Hand Signal
Time used = 157.64 milliseconds


# SVM

## Preprocessing

In [6]:
def svm_load_and_preprocess_subject_data(subject_id, base_path):
    mne.set_log_level("ERROR")
    mi_runs = [4, 8, 12]

    raw_files = []
    for run_id in mi_runs:
        subject_str = f"S{subject_id:03d}"
        run_str = f"R{run_id:02d}"
        file_path = f"{base_path}/{subject_str}/{subject_str}{run_str}.edf"

        if os.path.exists(file_path):
            try:
                raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
                raw_files.append(raw)
            except Exception as e:
                print(f"-- error to load {file_path}: {e}")
        else:
            pass

    if not raw_files:
        return None, None

    raw_concat = mne.concatenate_raws(raw_files, verbose=False)

    raw_concat.filter(config.LOWCUT, config.HIGHCUT, fir_design='firwin', skip_by_annotation='edge', verbose=False)
    raw_concat.notch_filter(freqs=50, verbose=False)
    # raw_concat.resample(config.RESAMPLE, npad="auto")
    raw_concat.set_eeg_reference('average', projection=False)
    raw_concat.apply_proj(verbose=False)

    events, event_id = mne.events_from_annotations(raw_concat, verbose=False)
    # print(events)

    # picks = mne.pick_types(raw_concat.info, meg=False, eeg=True, stim=False, eog=False, exclude='bads')
    picks = mne.pick_channels(raw_concat.info["ch_names"], include=config.SELECTED_CHANNELS)
    
    # Epoching with rejection
    reject_criteria = dict(eeg=200e-6)  
    flat_criteria = dict(eeg=1e-6)
        
    try:
        rest_epochs = mne.Epochs(raw_concat, events, dict(rest=1), 
                         tmin=config.T_MIN, tmax=config.T_MAX,
                         picks=picks, baseline=None, preload=True,
                         reject=reject_criteria, flat=flat_criteria,
                         on_missing='warn')
        epochs = mne.Epochs(raw_concat, events, dict(left_hand=2, right_hand=3),
                            tmin=config.T_MIN, tmax=config.T_MAX, proj=True,
                            picks=picks, baseline=(0.0, 1.0), preload=True, verbose=False, reject=reject_criteria, flat=flat_criteria,
                            on_missing='warn')
    except ValueError:
        return None, None
    
    if len(epochs)<=0:
        return None, None
    rest_data = rest_epochs.get_data()
    rest_mean = rest_data.mean(axis=(0, 2))

    # Normalize data - online vs offline
    X = epochs.get_data()
    X = X - rest_mean[:, np.newaxis] # baseline correction
    X = (X - X.mean(axis=-1, keepdims=True)) / X.std(axis=-1, keepdims=True)

    labels = epochs.events[:, -1] - 2  # Map labels to 0 (left) and 1 (right)
    X_fe = svm_feature_extract2(X) # 62.69%
    
    return X_fe, labels

## Energy and entropy based feature extraction

In [7]:
import numpy as np
from scipy.stats import entropy

def svm_feature_extract2(X, bins=32):
    """
    Extract features (Energy + Shannon Entropy) from EEG epochs.

    Parameters
    ----------
    X : ndarray
        Shape (n_epochs, n_channels, n_times)
    bins : int
        Number of bins for histogram (entropy)

    Returns
    -------
    X_fe : ndarray
        Shape (n_epochs, n_channels * 2)
        For each channel: [Energy, Entropy]
    """
    n_epochs, n_channels, n_times = X.shape
    X_fe = np.zeros((n_epochs, n_channels * 2))

    for i in range(n_epochs):
        features = []
        for ch in range(n_channels):
            signal = X[i, ch, :]

            # Energy
            energy = np.sum(signal ** 2)

            # Shannon entropy
            hist, _ = np.histogram(signal, bins=bins, density=True)
            hist = hist + 1e-12  # avoid log(0)
            shannon_ent = entropy(hist, base=2)

            features.extend([energy, shannon_ent])

        X_fe[i, :] = features

    return X_fe

In [8]:
def svm_load_dataset_for_subjects(subject_list, base_path):
    all_epochs_list = []
    all_labels_list = []
    # expected_n_samples = 641

    for subject_id in subject_list:
        print(f"Processing {subject_id:03d}...", end=" -> ")
        epochs, labels = svm_load_and_preprocess_subject_data(subject_id, base_path)

        if epochs is not None and len(epochs) > 0:
            # if expected_n_samples is None:
            #     expected_n_samples = epochs.shape[2]
            # if epochs.shape[2] != expected_n_samples:
            #     print(f"-- Wrong length ({epochs.shape[2]}). Skipping subject.")
            #     continue

            all_epochs_list.append(epochs)
            all_labels_list.append(labels)
            print(f"Finish {len(epochs)} trials.")
        else:
            print("-- No valid trials found. (dropped all)")

    if not all_epochs_list:
        return np.array([]), np.array([])

    return np.concatenate(all_epochs_list, axis=0), np.concatenate(all_labels_list, axis=0)

In [9]:
print("\n--- Loading Training Data ---")
X_train_svm, y_train_svm = svm_load_dataset_for_subjects(config.TRAIN_SVM_SUBJECTS, config.BASE_PATH)

print("\n--- Loading Testing Data ---")
X_test_svm, y_test_svm = svm_load_dataset_for_subjects(config.TEST_SVM_SUBJECTS, config.BASE_PATH)

if X_train_svm.size > 0 and X_test_svm.size > 0:
    print("\n Data Summary >> ")
    print(f" Training data shape: {X_train_svm.shape}")
    print(f" Training labels shape: {y_train_svm.shape} (Left: {np.sum(y_train_svm == 0)}, Right: {np.sum(y_train_svm == 1)})\")")
    print(f" Testing data shape: {X_test_svm.shape}")
    print(f" Testing labels shape: {y_test_svm.shape} (Left: {np.sum(y_test_svm == 0)}, Right: {np.sum(y_test_svm == 1)})\")")
else:
    print("\n-- Error")


--- Loading Training Data ---
Processing 001... -> Finish 41 trials.
Processing 002... -> Finish 45 trials.
Processing 003... -> Finish 41 trials.
Processing 004... -> Finish 45 trials.
Processing 005... -> Finish 45 trials.
Processing 006... -> Finish 41 trials.
Processing 007... -> Finish 39 trials.
Processing 008... -> Finish 45 trials.
Processing 009... -> -- No valid trials found. (dropped all)
Processing 010... -> Finish 42 trials.
Processing 011... -> Finish 44 trials.
Processing 012... -> Finish 45 trials.
Processing 013... -> -- No valid trials found. (dropped all)
Processing 014... -> Finish 45 trials.
Processing 015... -> Finish 28 trials.
Processing 016... -> Finish 45 trials.
Processing 017... -> Finish 8 trials.
Processing 018... -> Finish 45 trials.
Processing 019... -> Finish 16 trials.
Processing 020... -> Finish 45 trials.
Processing 021... -> Finish 45 trials.
Processing 022... -> Finish 35 trials.
Processing 023... -> Finish 16 trials.
Processing 024... -> -- No va

C:\Users\scpwk\AppData\Local\Temp\ipykernel_22304\2227600763.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\scpwk\AppData\Local\Temp\ipykernel_22304\2227600763.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\scpwk\AppData\Local\Temp\ipykernel_22304\2227600763.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.



Finish 13 trials.
Processing 101... -> Finish 45 trials.
Processing 102... -> Finish 42 trials.
Processing 103... -> Finish 14 trials.
Processing 104... -> Finish 42 trials.
Processing 105... -> Finish 45 trials.
Processing 106... -> Finish 37 trials.
Processing 107... -> Finish 37 trials.
Processing 108... -> Finish 7 trials.
Processing 109... -> Finish 44 trials.

 Data Summary >> 
 Training data shape: (3134, 28)
 Training labels shape: (3134,) (Left: 1588, Right: 1546)")
 Testing data shape: (713, 28)
 Testing labels shape: (713,) (Left: 353, Right: 360)")


## Baseline

In [38]:
def baseline_train_model(c, gamma):
    svm_model = SVC(
        C=c,
        kernel='linear',
        gamma=gamma,
        random_state=42,
        # class_weight='balanced',
        verbose=True
    )
    
    svm_model.fit(X_train_svm, y_train_svm)

    y_train_pred = svm_model.predict(X_train_svm)
    y_pred = svm_model.predict(X_test_svm)
    
    loss_fn = tf.keras.losses.Hinge()
    
    loss_train = loss_fn(y_train_svm, y_train_pred)
    loss_test = loss_fn(y_test_svm, y_pred)
    
    acc_train = accuracy_score(y_train_svm, y_train_pred)
    acc_test = accuracy_score(y_test_svm, y_pred)
    
    print(f"\n--- for c = {c} and gamma = {gamma} ---")
    print("Train Accuracy:", acc_train)
    print("   - Train Loss: ", loss_train)
    print("   - Test Loss:", loss_test)
    print("Test Accuracy:", acc_test)
    print("\n--- Classification Report (Test Set) ---")
    print(classification_report(y_test_svm, y_pred, target_names=['Left Hand', 'Right Hand']))
    plot_confusion_matrix(y_test_svm, y_pred)

In [39]:
baseline_train_model(120000, 'scale')

[LibSVM]
--- for c = 120000 and gamma = scale ---
Train Accuracy: 0.6059349074664965
   - Train Loss:  tf.Tensor(0.9007658, shape=(), dtype=float32)
   - Test Loss: tf.Tensor(0.88218796, shape=(), dtype=float32)
Test Accuracy: 0.6129032258064516

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

   Left Hand       0.60      0.65      0.62       353
  Right Hand       0.63      0.57      0.60       360

    accuracy                           0.61       713
   macro avg       0.61      0.61      0.61       713
weighted avg       0.61      0.61      0.61       713



## Training

In [10]:
def train_model(c, gamma):
    svm_model = SVC(
        C=c,
        kernel='rbf',
        gamma=gamma,
        random_state=42,
        probability=True,
        verbose=True
    )
    
    svm_model.fit(X_train_svm, y_train_svm)

    y_train_pred = svm_model.predict(X_train_svm)
    y_pred = svm_model.predict(X_test_svm)
    
    # y_proba_train = svm_model.predict_proba(X_train_svm)
    # y_proba_test = svm_model.predict_proba(X_test_svm)
    
    cal_loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)
    loss_train = cal_loss(y_train_svm, y_train_pred)
    loss_test = cal_loss(y_test_svm, y_pred)
    
    acc_train = accuracy_score(y_train_svm, y_train_pred)
    acc_test = accuracy_score(y_test_svm, y_pred)
    
    print(f"\n--- for c = {c} and gamma = {gamma} ---")
    print("Train Accuracy:", acc_train)
    print('   - Train Loss:', loss_train)
    print("Test Accuracy:", acc_test)
    print('   - Test Loss:', loss_test)
    print("\n--- Classification Report (Test Set) ---")
    print(classification_report(y_test_svm, y_pred, target_names=['Left Hand', 'Right Hand']))
    plot_confusion_matrix(y_test_svm, y_pred)
    
    return svm_model

In [11]:
svm_model = train_model(120000, 'scale')

[LibSVM]
--- for c = 120000 and gamma = scale ---
Train Accuracy: 0.6186981493299298
   - Train Loss: tf.Tensor(0.6961325, shape=(), dtype=float32)
Test Accuracy: 0.6269284712482468
   - Test Loss: tf.Tensor(0.6858539, shape=(), dtype=float32)

--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

   Left Hand       0.62      0.64      0.63       353
  Right Hand       0.64      0.61      0.62       360

    accuracy                           0.63       713
   macro avg       0.63      0.63      0.63       713
weighted avg       0.63      0.63      0.63       713



In [132]:
m = int(input('Input Index : '))

start_time = time.time()

pred_svm = svm_model.predict_proba(X_test_svm[m:m+1])

end_time = time.time()

real_num_svm = np.argmax(y_test_svm[m])
pred_one(pred_svm, real_num_svm)

fin_time = end_time - start_time
print(f"Time used = {fin_time*1000:.2f} milliseconds")

# svm_model.predict(X_test_svm[m:m+1])

[[0.41565969 0.58434031]]

-----------------------------------

Predicted Class :  Right Hand Signal
Accuracy : 58.43%
Real Class :  Left Hand Signal
Time used = 1.00 milliseconds
